
**Project:** E-Commerce Customer Intelligence & Revenue Growth Platform

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("CSV files:", len(list(RAW_DIR.glob("*.csv"))))

Raw folder: C:\Users\dharmveer patel\OneDrive\Desktop\ecommerce_customer_intelligence_major_project_complete\ecommerce_customer_intelligence_major_project\data\raw
CSV files: 9


In [2]:
file_map = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tables = {name: pd.read_csv(RAW_DIR / filename) for name, filename in file_map.items()}

for name, df in tables.items():
    print(f"{name:20s} rows={len(df):>8,} | columns={df.shape[1]:>2}")

orders               rows=  99,441 | columns= 8
customers            rows=  99,441 | columns= 5
items                rows= 112,650 | columns= 7
payments             rows= 103,886 | columns= 5
reviews              rows=  99,224 | columns= 7
products             rows=  32,951 | columns= 9
sellers              rows=   3,095 | columns= 4
category_translation rows=      71 | columns= 2


In [3]:
audit = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_full_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_pct_of_cells": round(df.isna().sum().sum() / max(df.size, 1) * 100, 2),
    }
    for name, df in tables.items()
]).sort_values("rows", ascending=False)

display(audit)
audit.to_csv(OUTPUT_DIR / "data_audit_summary.csv", index=False)

,table,rows,columns,duplicate_full_rows,missing_cells,missing_pct_of_cells
2,items,112650,7,0,0,0.00
3,payments,103886,5,0,0,0.00
1,customers,99441,5,0,0,0.00
0,orders,99441,8,0,4908,0.62
4,reviews,99224,7,0,145903,21.01
5,products,32951,9,0,2448,0.83
6,sellers,3095,4,0,0,0.00
7,category_translation,71,2,0,0,0.00


In [4]:
key_specs = {
    "orders": ["order_id", "customer_id"],
    "customers": ["customer_id", "customer_unique_id"],
    "items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "payments": ["order_id", "payment_sequential"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "reviews": ["review_id", "order_id"],
}

key_results = []
for table_name, columns in key_specs.items():
    df = tables[table_name]
    for col in columns:
        key_results.append({
            "table": table_name,
            "column": col,
            "missing": int(df[col].isna().sum()),
            "distinct": int(df[col].nunique(dropna=True)),
            "rows": len(df),
            "duplicate_values": int(df[col].duplicated().sum()),
        })

key_audit = pd.DataFrame(key_results)
display(key_audit)
key_audit.to_csv(OUTPUT_DIR / "key_audit.csv", index=False)

,table,column,missing,distinct,rows,duplicate_values
0,orders,order_id,0,99441,99441,0
1,orders,customer_id,0,99441,99441,0
2,customers,customer_id,0,99441,99441,0
3,customers,customer_unique_id,0,96096,99441,3345
4,items,order_id,0,98666,112650,13984
5,items,order_item_id,0,21,112650,112629
6,items,product_id,0,32951,112650,79699
7,items,seller_id,0,3095,112650,109555
8,payments,order_id,0,99440,103886,4446
9,payments,payment_sequential,0,29,103886,103857


In [5]:
orders = tables["orders"].copy()
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("Missing purchase timestamp:", int(orders["order_purchase_timestamp"].isna().sum()))
print("Order statuses:")
display(orders["order_status"].value_counts(dropna=False).to_frame("orders"))

Missing purchase timestamp: 0
Order statuses:


,orders
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [7]:
customers = tables["customers"].copy()
items = tables["items"].copy()
payments = tables["payments"].copy()
reviews = tables["reviews"].copy()
products = tables["products"].copy()
sellers = tables["sellers"].copy()
category_translation = tables["category_translation"].copy()

# Trim accidental whitespace from string columns where relevant.
for df in [orders, customers, items, payments, reviews, products, sellers, category_translation]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

# Save prepared tables
clean_tables = {
    "orders": orders,
    "customers": customers,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}
for name, df in clean_tables.items():
    df.to_csv(CLEAN_DIR / f"{name}.csv", index=False)

print("Saved prepared tables to:", CLEAN_DIR)

Saved prepared tables to: C:\Users\dharmveer patel\OneDrive\Desktop\ecommerce_customer_intelligence_major_project_complete\ecommerce_customer_intelligence_major_project\data\processed
